### Power Network and Statistical Package

In [53]:
import os, shutil, random
import numpy as np
import networkx as nx
import pandas as pd
import torch
import copy
import cvxpy as cp

#Pandapower Package
import pandapower as pp
import pandapower.networks as pn
import pandapower.plotting as plot
import pandapower.diagnostic as diagnostic

from pandapower.powerflow import LoadflowNotConverged
from pandapower.diagnostic import diagnostic
from pandapower.control import ConstControl
from pandapower.timeseries import DFData, OutputWriter, run_timeseries
from pandapower.pypower.makeYbus import makeYbus

import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.linalg
import scipy.sparse as sp
from scipy.stats import gaussian_kde
from itertools import product
from tqdm import tqdm

In [55]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### Define the Function to check N-1 contingency criterion with the test case and reinforce it meet the criterion

In [58]:
# -*- coding: utf-8 -*-
"""
RTS-24 with renewables + N-1 reinforcement (root-cause aware)
- Adds renewables (optional) with your Pmax (from real data)
- Per-element cost rows (avoid pandas vectorized assignment error)
- N-1 screening (line/trafo/gen outages) via AC-OPF
- Reinforcement policy:
    * If the outage causes OPF infeasibility:
        - line outage -> add a parallel line to THAT line
        - trafo outage -> add a parallel transformer to THAT transformer
        - gen outage -> increase max_p_mw for OTHER generators slightly
    * If outage is feasible but has line overloads -> add parallel lines to overloaded lines
- Iterate until N-1 passes or max iterations reached
- Saves final network as JSON
Tested with: Python 3.11, pandapower 2.13+

"""
# -----------------------
# (Optional) renewables from the real data
# Fill bus ID and Pmax (MW). If not needed, leave the list empty.
RENEWABLE_PLANTS = [
    # (bus_id_in_rts24, Pmax_MW)
    # (3, 80.0),
    # (7, 60.0),
    # (16, 50.0),
]

# -----------------------
# Global settings
# -----------------------
VMIN_PU = 0.95
VMAX_PU = 1.05
LINE_LOADING_LIMIT = 100.0    # %
MAX_ITER = 16                 # allow more passes
OPF_TOL = 1e-6
ADD_PARALLEL_LINES = True     # True: add parallel circuits; False: uprate ampacity
LINE_UPRATE_FACTOR = 1.25     # used when not adding parallels
GEN_MAXP_UPFACTOR  = 1.05     # every time we relax gens

# Slack pricier than gens -> prefer local gens first (more realistic)
UNLIMITED_SLACK = False

# (Optional) caps to avoid infinite growth
MAX_PARALLELS_PER_ELEMENT = 3

# -----------------------
# Utilities
# -----------------------
def add_cost_row_per_element(net, element_indices, et, cp0=0.0, cp1=5.0, cp2=0.02):
    """ Add poly cost rows one-by-one (avoid vectorized create_poly_cost)."""
    for idx in element_indices:
        pp.create_poly_cost(
            net, element=int(idx), et=et,
            cp0_eur=cp0, cp1_eur_per_mw=cp1, cp2_eur_per_mw2=cp2
        )

def build_base_rts24():
    net = pn.case24_ieee_rts()

    # Voltage bounds
    net.bus["min_vm_pu"] = VMIN_PU
    net.bus["max_vm_pu"] = VMAX_PU

    # Ensure controllable flags
    for df in (net.gen, net.ext_grid):
        if "controllable" not in df.columns:
            df["controllable"] = True
        else:
            df["controllable"] = True

    # Wide bounds for ext_grid (feasibility guard); cost decides dispatch preference
    for col, val in [("min_p_mw", -1e4), ("max_p_mw", 1e4),
                     ("min_q_mvar", -1e4), ("max_q_mvar", 1e4)]:
        if col not in net.ext_grid.columns:
            net.ext_grid[col] = val
        else:
            net.ext_grid[col] = val

    # Ensure generator bounds exist
    if "min_p_mw" not in net.gen.columns:
        net.gen["min_p_mw"] = 0.0
    if "max_p_mw" not in net.gen.columns:
        net.gen["max_p_mw"] = np.maximum(net.gen["p_mw"].values * 1.5, 20.0)
    if "min_q_mvar" not in net.gen.columns:
        net.gen["min_q_mvar"] = -1e3
    if "max_q_mvar" not in net.gen.columns:
        net.gen["max_q_mvar"] = 1e3

    # Ensure line ampacity exists
    if "max_i_ka" not in net.line.columns:
        net.line["max_i_ka"] = 1.0

    # Track how many parallels we added (custom columns)
    if "parallels_added" not in net.line.columns:
        net.line["parallels_added"] = 0
    if "parallels_added" not in net.trafo.columns:
        net.trafo["parallels_added"] = 0

    # Reset poly_cost, then add per-element rows
    if len(net.poly_cost):
        net.poly_cost.drop(net.poly_cost.index, inplace=True)

    # Conventional gens: normal cost
    if len(net.gen):
        add_cost_row_per_element(net, net.gen.index.tolist(), et="gen", cp0=0.0, cp1=5.0, cp2=0.02)

    # External grid: pricier than gens (prefer local gen first)
    if len(net.ext_grid):
        add_cost_row_per_element(net, net.ext_grid.index.tolist(), et="ext_grid",
                                 cp0=0.0,
                                 cp1=(1e-6 if UNLIMITED_SLACK else 8.0),
                                 cp2=(0.0   if UNLIMITED_SLACK else 0.02))
    return net

def add_renewables(net, plants):
    """ Add renewable generators at buses with given Pmax; very low marginal cost."""
    new_gen_indices = []
    for bus, pmax in plants:
        gidx = pp.create_gen(
            net, bus=int(bus), p_mw=0.0, vm_pu=1.0,
            min_p_mw=0.0, max_p_mw=float(pmax), name=f"RES@{bus}"
        )
        new_gen_indices.append(gidx)
    if new_gen_indices:
        # cheap renewable cost
        add_cost_row_per_element(net, new_gen_indices, et="gen", cp0=0.0, cp1=0.05, cp2=0.0)


def run_ac_opf(net):
    try:
        pp.runopp(net, calculate_voltage_angles=True, verbose=False, enforce_q_lims=True, delta=OPF_TOL)
        return True, ""
    except Exception as e:
        return False, f"OPF failed: {e}"


def check_security_criteria(net):
    vm = net.res_bus.vm_pu
    volt_ok = bool((vm >= VMIN_PU - 1e-6).all() and (vm <= VMAX_PU + 1e-6).all())
    if len(net.res_line):
        loading = net.res_line.loading_percent.fillna(0.0)
        line_ok = bool((loading <= LINE_LOADING_LIMIT + 1e-6).all())
    else:
        line_ok = True
    return volt_ok and line_ok


class OutageCtx:
    def __init__(self, net, table, idx): self.net, self.table, self.idx = net, table, idx
    def __enter__(self): getattr(self.net, self.table).at[self.idx, "in_service"] = False
    def __exit__(self, exc_type, exc, tb):
        getattr(self.net, self.table).at[self.idx, "in_service"] = True
        return False


def add_parallel_line_like(net, lid):
    """Add a parallel line with same electrical parameters or std_type."""
    if lid not in net.line.index:
        return None
    ln = net.line.loc[lid]
    # Avoid infinite growth
    if net.line.at[lid, "parallels_added"] >= MAX_PARALLELS_PER_ELEMENT:
        # if cap reached, try uprating instead
        net.line.at[lid, "max_i_ka"] = float(ln.max_i_ka) * LINE_UPRATE_FACTOR
        return "uprated"
    if "std_type" in net.line.columns and isinstance(ln.get("std_type"), str) and ln.std_type:
        new_id = pp.create_line(
            net, from_bus=int(ln.from_bus), to_bus=int(ln.to_bus),
            length_km=float(ln.length_km), std_type=str(ln.std_type),
            name=f"parallel_of_line_{lid}"
        )
    else:
        new_id = pp.create_line_from_parameters(
            net, from_bus=int(ln.from_bus), to_bus=int(ln.to_bus),
            length_km=float(ln.length_km),
            r_ohm_per_km=float(ln.r_ohm_per_km),
            x_ohm_per_km=float(ln.x_ohm_per_km),
            c_nf_per_km=float(ln.c_nf_per_km),
            max_i_ka=float(ln.max_i_ka),
            name=f"parallel_of_line_{lid}"
        )
    net.line.at[lid, "parallels_added"] += 1
    return new_id


def add_parallel_trafo_like(net, tid):
    """Add a parallel transformer with same std_type or parameters."""
    if tid not in net.trafo.index:
        return None
    tr = net.trafo.loc[tid]
    # Avoid infinite growth
    if net.trafo.at[tid, "parallels_added"] >= MAX_PARALLELS_PER_ELEMENT:
        # fallback: increase trafo sn_mva (only if create_from_parameters path used)
        if "sn_mva" in net.trafo.columns:
            net.trafo.at[tid, "sn_mva"] = float(tr.sn_mva) * 1.20
        return "uprated"
    if "std_type" in net.trafo.columns and isinstance(tr.get("std_type"), str) and tr.std_type:
        new_id = pp.create_transformer(
            net, hv_bus=int(tr.hv_bus), lv_bus=int(tr.lv_bus),
            std_type=str(tr.std_type), name=f"parallel_of_trafo_{tid}"
        )
    else:
        # Create from parameters (need core params)
        kwargs = dict(
            hv_bus=int(tr.hv_bus), lv_bus=int(tr.lv_bus),
            sn_mva=float(tr.sn_mva),
            vn_hv_kv=float(tr.vn_hv_kv), vn_lv_kv=float(tr.vn_lv_kv),
            vk_percent=float(getattr(tr, "vk_percent", 10.0)),
            vkr_percent=float(getattr(tr, "vkr_percent", 0.3)),
            pfe_kw=float(getattr(tr, "pfe_kw", 0.0)),
            i0_percent=float(getattr(tr, "i0_percent", 0.0)),
            name=f"parallel_of_trafo_{tid}"
        )
        new_id = pp.create_transformer_from_parameters(net, **kwargs)
    net.trafo.at[tid, "parallels_added"] += 1
    return new_id


def n1_report(net):
    """Run N-1 screening and record violations."""
    rep = {"violations": [], "passed": True}

    def record(kind, idx, detail):
        rep["violations"].append({"type": kind, "index": int(idx), "detail": detail})
        rep["passed"] = False

    line_ids  = list(net.line.index)
    gen_ids   = list(net.gen.index)
    trafo_ids = list(net.trafo.index) if "trafo" in net and len(net.trafo) else []

    # Lines
    for lid in line_ids:
        with OutageCtx(net, "line", lid):
            ok, msg = run_ac_opf(net)
            if not ok:
                record("line", lid, f"OPF infeasible. {msg}")
                continue
            if not check_security_criteria(net):
                overloaded = net.res_line[net.res_line.loading_percent > LINE_LOADING_LIMIT].index.tolist()
                vmin = float(net.res_bus.vm_pu.min()); vmax = float(net.res_bus.vm_pu.max())
                record("line", lid, f"Overloads: {overloaded}; vm_pu [{vmin:.3f},{vmax:.3f}]")

    # Generators
    for gid in gen_ids:
        with OutageCtx(net, "gen", gid):
            ok, msg = run_ac_opf(net)
            if not ok:
                record("gen", gid, f"OPF infeasible. {msg}")
                continue
            if not check_security_criteria(net):
                overloaded = net.res_line[net.res_line.loading_percent > LINE_LOADING_LIMIT].index.tolist()
                vmin = float(net.res_bus.vm_pu.min()); vmax = float(net.res_bus.vm_pu.max())
                record("gen", gid, f"Overloads: {overloaded}; vm_pu [{vmin:.3f},{vmax:.3f}]")

    # Transformers
    for tid in trafo_ids:
        with OutageCtx(net, "trafo", tid):
            ok, msg = run_ac_opf(net)
            if not ok:
                record("trafo", tid, f"OPF infeasible. {msg}")
                continue
            if not check_security_criteria(net):
                overloaded = net.res_line[net.res_line.loading_percent > LINE_LOADING_LIMIT].index.tolist()
                vmin = float(net.res_bus.vm_pu.min()); vmax = float(net.res_bus.vm_pu.max())
                record("trafo", tid, f"Overloads: {overloaded}; vm_pu [{vmin:.3f},{vmax:.3f}]")

    return rep


def parse_overloaded_lines(detail):
    """Extract overloaded line IDs from violation detail text."""
    try:
        part = detail.split("Overloads:", 1)[1]
        arr  = part.split("[", 1)[1].split("]", 1)[0]
        ids = []
        for t in arr.split(","):
            t = t.strip()
            if t:
                ids.append(int(t))
        return ids
    except Exception:
        return []


def reinforce_once(net, violations):
    """
    Root-cause aware reinforcement:
      - If OPF infeasible due to *line outage*: add parallel to that line
      - If OPF infeasible due to *trafo outage*: add parallel to that trafo
      - If OPF infeasible due to *gen outage*: relax OTHER gens' max_p
      - If feasible but overloaded lines: add a parallel line to those overloaded lines (or uprate)
    Returns the number of changes applied.
    """
    changes = 0
    lines_to_parallel_from_overload = set()
    need_expand_gens = False

    for v in violations:
        t, idx, detail = v["type"], v["index"], v["detail"]

        if "OPF infeasible" in detail:
            if t == "line":
                res = add_parallel_line_like(net, idx)
                changes += 1 if res is not None else 0
            elif t == "trafo":
                res = add_parallel_trafo_like(net, idx)
                changes += 1 if res is not None else 0
            elif t == "gen":
                need_expand_gens = True

        elif "Overloads:" in detail:
            for lid in parse_overloaded_lines(detail):
                lines_to_parallel_from_overload.add(lid)

    # Handle overloads (feasible cases)
    for lid in lines_to_parallel_from_overload:
        if ADD_PARALLEL_LINES:
            res = add_parallel_line_like(net, lid)
            changes += 1 if res is not None else 0
        else:
            if lid in net.line.index:
                net.line.at[lid, "max_i_ka"] = float(net.line.at[lid, "max_i_ka"]) * LINE_UPRATE_FACTOR
                changes += 1

    # Handle gen expansion (exclude outaged one by nature of context)
    if need_expand_gens and len(net.gen):
        net.gen["max_p_mw"] *= GEN_MAXP_UPFACTOR
        changes += 1

    return changes


def reinforce_until_n1_secure(net, max_iter=MAX_ITER):
    for it in range(1, max_iter + 1):
        # Ensure base case is feasible (slightly relax gens if needed)
        ok, _ = run_ac_opf(net)
        if not ok or not check_security_criteria(net):
            net.gen["max_p_mw"] *= GEN_MAXP_UPFACTOR

        rep = n1_report(net)
        if rep["passed"]:
            return True, rep, net

        changed = reinforce_once(net, rep["violations"])
        if changed == 0:
            # No actionable changes; stop early
            return False, rep, net

    # Final check after reaching iteration cap
    rep = n1_report(net)
    return rep["passed"], rep, net


def pretty_summary(rep, limit=20):
    if rep["passed"]:
        return " N-1 TEST PASSED."
    out = ["  N-1 TEST NOT PASSED. Sample violations:"]
    for v in rep["violations"][:limit]:
        out.append(f"  - outage type={v['type']}, idx={v['index']}: {v['detail']}")
    return "\n".join(out)

# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    # 1) Base RTS-24
    base = build_base_rts24()

    # 2) Add renewables from your real data (optional)
    if RENEWABLE_PLANTS:
        add_renewables(base, RENEWABLE_PLANTS)

    # 3) Deep copy & reinforce
    net = copy.deepcopy(base)
    ok, report, reinforced = reinforce_until_n1_secure(net, max_iter=MAX_ITER)
    print(pretty_summary(report))

    # 4) Save
    out_file = "rts24_n1_reinforced_with_res_rootcause.json"
    pp.to_json(reinforced, out_file)
    print(f"Saved: {out_file}")

 N-1 TEST PASSED.
Saved: rts24_n1_reinforced_with_res_rootcause.json


In [59]:
net = pp.from_json("rts24_n1_reinforced_with_res_rootcause.json")

print(net)

This pandapower network includes the following parameter tables:
   - bus (24 elements)
   - load (17 elements)
   - sgen (22 elements)
   - gen (10 elements)
   - shunt (1 element)
   - ext_grid (1 element)
   - line (43 elements)
   - trafo (6 elements)
   - poly_cost (11 elements)
   - bus_geodata (24 elements)
 and the following results tables:
   - res_bus (24 elements)
   - res_line (43 elements)
   - res_trafo (6 elements)
   - res_ext_grid (1 element)
   - res_load (17 elements)
   - res_sgen (22 elements)
   - res_shunt (1 element)
   - res_gen (10 elements)
 and the following result values:
   - res_cost


In [60]:
# 1) Total number of lines / Number of lines reinforced
print("Total lines:", len(net.line))

print("Reinforced lines (parallels_added > 0):", (net.line["parallels_added"] > 0).sum())

# 2) List the reinforced lines
reinforced_lines = net.line[net.line["parallels_added"] > 0]

print(reinforced_lines[["from_bus", "to_bus", "parallels_added", "max_i_ka", "name"]])

# 3) Transformer is handled similarly
if "trafo" in net and len(net.trafo):
    print("Total trafos:", len(net.trafo))
    print("Reinforced trafos:", (net.trafo["parallels_added"] > 0).sum())
    reinforced_trafos = net.trafo[net.trafo["parallels_added"] > 0] 
    print(reinforced_trafos[["hv_bus", "lv_bus", "parallels_added", "sn_mva", "name"]])

Total lines: 43
Reinforced lines (parallels_added > 0): 6
    from_bus  to_bus  parallels_added  max_i_ka  name
3          1       3              1.0  0.732147  None
4          1       5              2.0  0.732147  None
8          5       9              1.0  0.732147  None
9          6       7              2.0  0.732147  None
11         7       9              1.0  0.732147  None
21        14      23              1.0  1.255109  None
Total trafos: 6
Reinforced trafos: 1
   hv_bus  lv_bus  parallels_added  sn_mva  name
0      23       2              1.0   400.0  None


In [61]:
print(net.gen[["bus", "p_mw", "max_p_mw", "name"]])
total_cap = net.gen["max_p_mw"].sum()
print(f"Total generation capacity (max_p_mw sum) = {total_cap:.2f} MW")

   bus   p_mw  max_p_mw  name
0    0   10.0   22.0500  None
1    1   10.0   22.0500  None
2    6   80.0  110.2500  None
3   13    0.0    0.0000  None
4   14   12.0   13.2300  None
5   15  155.0  170.8875  None
6   17  400.0  441.0000  None
7   20  400.0  441.0000  None
8   21   50.0   55.1250  None
9   22  155.0  170.8875  None
Total generation capacity (max_p_mw sum) = 1446.48 MW


In [ ]:
## Built PMU with the Loaded Network Grid
# def get_single_bus_pmu_measurement(net, bus_index):
#     """
#     Extract PMU measurement for a single bus:
#     - Voltage magnitude (p.u.)
#     - Voltage angle (degrees)
#     - Active power injection (MW)
#     - Reactive power injection (MVAR)
    
#     Returns a dictionary with all relevant fields.
#     """
#     vm = float(net.res_bus.vm_pu.at[bus_index])
#     va = float(net.res_bus.va_degree.at[bus_index])
#     p = float(net.res_bus.p_mw.at[bus_index])
#     q = float(net.res_bus.q_mvar.at[bus_index])
    
#     return {
#         'bus': bus_index,
#         'vm_pu': vm,
#         'va_degree': va,
#         'p_mw': p,
#         'q_mvar': q
#     }


# def pdc(net, pmu_buses):
#     """
#     Simulate the Phasor Data Concentrator (PDC):
#     Collects PMU data from all buses listed in `pmu_buses`.
#     Returns a list of measurement dictionaries.
#     """
#     pmu_data = []
#     for bus in pmu_buses:
#         measurement = get_single_bus_pmu_measurement(net, bus)
#         pmu_data.append(measurement)
#     return pmu_data


# # Example usage
# pmu_buses = net.bus.index.tolist()  # simulate PMU on all buses
# pmu_data_frame = pdc(net, pmu_buses)

# # Optional: convert to a table (like your screenshot)
# import pandas as pd
# pmu_df = pd.DataFrame(pmu_data_frame)
# print(pmu_df)

In [ ]:
 # net.res_bus

### Information Layer Construct

In [ ]:
# # Define sensors at selected buses
# pmu_buses = [1, 3, 7, 15]
# sensor_data = {}

# for bus in pmu_buses:
#     sensor_data[f"Bus-{bus}"] = {
#         "type": "PMU",
#         "vm_pu_true": None,
#         "vm_pu_measured": None,
#         "measurement_delay": 1,  # in timesteps
#         "communication_error": 0.01  # 1% measurement noise
#     }

In [ ]:
# def simulate_info_layer(net, sensor_data):
#     for bus_tag, data in sensor_data.items():
#         bus_idx = int(bus_tag.split("-")[1])
#         true_value = net.res_bus.vm_pu.at[bus_idx]
#         noise = np.random.normal(0, data["communication_error"])
#         sensor_data[bus_tag]["vm_pu_true"] = true_value
#         sensor_data[bus_tag]["vm_pu_measured"] = true_value + noise

### Apply the cyber attack by injecting false data

In [ ]:
# def inject_false_data(pmu_data, attack_buses, voltage_offset=0.05, angle_offset=2.0):
#     """
#     Inject false data at specified bus indices.
#     voltage_offset: additive error in per unit (p.u.)
#     angle_offset: additive error in degrees
#     """
#     pmu_data_attacked = pmu_data.copy()
#     # Create copies to avoid modifying original
#     voltage_attacked = np.copy(pmu_data_attacked['voltage'])
#     angle_attacked = np.copy(pmu_data_attacked['angle'])
    
#     for idx in attack_buses:
#         voltage_attacked[idx] += voltage_offset  # add offset
#         angle_attacked[idx] += angle_offset
#     pmu_data_attacked['voltage'] = voltage_attacked
#     pmu_data_attacked['angle'] = angle_attacked
#     return pmu_data_attacked

# # Assume attackers compromise bus 5, 12, and 23 
# attack_buses = [5, 12, 23]
# pmu_data_attacked = inject_false_data(pmu_data_true, attack_buses)

In [ ]:
# def compute_voltage_deviation(net, baseline_vm_pu):
#     deviations = abs(net.res_bus.vm_pu - baseline_vm_pu)
#     return deviations.mean()

In [ ]:
# def compute_custom_resilience_metric(system_loss, recovery_time, max_loss=1.0):
#     # Normalize values between 0 and 1
#     L_norm = system_loss / max_loss
#     T_norm = recovery_time / 10  # assume 10 is worst case

#     # Resilience = 1 - (weighted penalty)
#     return 1 - (0.6 * L_norm + 0.4 * T_norm)

###  Run Time-Series Simulations with Events
Define multiple time steps to simulate pre-attack, attack, recovery periods

In [ ]:
# T = 10
# baseline_vm_pu = None

# for t in range(T):
#     if t == 0:
#         pp.runpp(net)
#         baseline_vm_pu = net.res_bus.vm_pu.copy()

#     if t == 3:
#         apply_fdia(sensor_data, target_bus=7, offset=0.15)

#     if t == 5:
#         break_communication(sensor_data, bus_id=15)

#     simulate_info_layer(net, sensor_data)

#     # Log metrics
#     voltage_dev = compute_voltage_deviation(net, baseline_vm_pu)
#     print(f"t={t}, Voltage Deviation: {voltage_dev:.4f}")

## Resilience Metrics Evaluation

### The Resilience Metric
We quantify **physical resilience** via a time-varying score \(S_P(t)\) that aggregates three normalized indicators—load served ratio, voltage persistence, and restoration performance—using a generalized power mean, so that the score stays in \([0,1]\) and can emphasize worst-case or average behavior.  

In parallel, we define a **cyber resilience score** \(S_C(t)\) that combines cyber reachability \(C_t\) with detection and recovery times of cyber attacks, penalizing long detection/recovery delays through exponential terms.  

These two scores provide coupled physical–cyber metrics that can be tracked over time or used as objectives/constraints in our response and recovery optimization.


### Test case 1- line trip+PMU cluster loss

In [ ]:
# =====================================================================================
# 1. BASE-CASE PREP: in_service flags, gen limits, base power flow
# =====================================================================================

# Ensure in-service flags exist and are True
for table in ["bus", "line", "gen", "load", "ext_grid", "trafo"]:
    if table in net:
        if "in_service" not in net[table].columns:
            net[table]["in_service"] = True
        else:
            net[table]["in_service"] = net[table]["in_service"].fillna(True)

# Ensure generator min/max P exist and are reasonable
if "gen" in net:
    if "min_p_mw" not in net.gen.columns:
        net.gen["min_p_mw"] = 0.0  # can go down to 0 (or negative if desired)
    if "max_p_mw" not in net.gen.columns:
        net.gen["max_p_mw"] = net.gen["p_mw"].values * 1.2  # small headroom

# Test base-case power flow
pp.runpp(net)
print("Base case converged:", net.converged)


In [ ]:
# =====================================================================================
# 2. GLOBAL CONFIG & RESILIENCE PARAMETERS
#    Response phase: looser voltage, more shedding allowed.
#    Recovery phase: stricter voltage and higher load restoration.
# =====================================================================================

case_name   = "RTS24_resilience"
n_ts        = 96             # 24 hours @ 15-min resolution
step_hours  = 0.25           # 15 minutes
rng         = np.random.default_rng(0)

# Scenario and PMU configuration
NUM_SCENARIOS = 10
NUM_PMUS      = 6

# Weights for cyber score
u1 = 0.3   # weight for C_t
u2 = 0.25  # weight for T_det
u3 = 0.25  # weight for T_resp
u4 = 0.2   # weight for T_rec

# Physical & cyber constraints
gamma = 0.85       # minimum load service ratio in response (≥ 85% of total load)
epsilon = 0.10     # max voltage deviation in response (0.9–1.1 pu, emergency limits)

Beta = 0.85        # minimum cyber reachability in recovery (≥ 85% of control nodes)
Gamma = 0.95       # load service target in recovery (≥ 95% of nominal load)

epsilon_rec = 0.05 # recovery voltage band (0.95–1.05 pu, normal planning band)

# Time normalizations (design choices, in hours)
Tdet_max  = 6.0
Tresp_max = 4.0
Trec_max  = 12.0

# Case directory
base_root = f"./{case_name}"
os.makedirs(base_root, exist_ok=True)

# Store original network state
original_line_in_service  = net.line["in_service"].copy()
original_gen_in_service   = net.gen["in_service"].copy()  if "gen"  in net else None
original_load_in_service  = net.load["in_service"].copy() if "load" in net else None

# Scenario target element (here: line outages)
target_element  = "line"
element_indices = list(net[target_element].index)

# Time index (24 h, 15-min step)
time_index = pd.date_range("2025-01-01 00:00:00", periods=n_ts,
                           freq=pd.Timedelta(hours=step_hours))


In [ ]:
# =====================================================================================
# 3. HELPER FUNCTIONS: physical metrics, cyber network, optimization, scores
# =====================================================================================

def add_bus_information(net):
    """Attach static bus-related info (bus indices, in_service) to result tables."""
    if "res_line" in net and "line" in net:
        net.res_line = net.res_line.merge(
            net.line[["from_bus", "to_bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_line = net.res_line.loc[:, ~net.res_line.columns.str.endswith('_y')]

    if "res_gen" in net and "gen" in net:
        net.res_gen = net.res_gen.merge(
            net.gen[["bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_gen = net.res_gen.loc[:, ~net.res_gen.columns.str.endswith('_y')]

    if "res_load" in net and "load" in net:
        net.res_load = net.res_load.merge(
            net.load[["bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_load = net.res_load.loc[:, ~net.res_load.columns.str.endswith('_y')]

    if "res_ext_grid" in net and "ext_grid" in net:
        net.res_ext_grid = net.res_ext_grid.merge(
            net.ext_grid[["bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_ext_grid = net.res_ext_grid.loc[:, ~net.res_ext_grid.columns.str.endswith('_y')]

    if "res_trafo" in net and "trafo" in net:
        net.res_trafo = net.res_trafo.merge(
            net.trafo[["hv_bus", "lv_bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_trafo = net.res_trafo.loc[:, ~net.res_trafo.columns.str.endswith('_y')]

    return net


def compute_load_served_ratio_from_results(net):
    """LSR = sum of served load / total demand, based on buses connected via active lines."""
    if not ("res_load" in net and "load" in net and "line" in net):
        return np.nan

    active_lines = net.line[net.line["in_service"] == True]
    active_buses = set(active_lines["from_bus"]).union(set(active_lines["to_bus"]))

    if "bus" not in net.res_load.columns:
        net = add_bus_information(net)

    served_loads = net.res_load[net.res_load["bus"].isin(active_buses)]
    served = served_loads["p_mw"].sum()
    demand = net.res_load["p_mw"].sum()

    if demand <= 0:
        return np.nan

    return min(served / demand, 1.0)


def compute_voltage_deviation(net):
    """Maximum voltage deviation from 1.0 pu."""
    if "res_bus" not in net:
        return np.nan
    voltage_deviations = np.abs(net.res_bus["vm_pu"] - 1.0)
    return voltage_deviations.max()


def compute_voltage_persistence_from_results(net):
    """
    Voltage persistence factor based on the largest abnormal-voltage cluster.
    Larger abnormal cluster => lower persistence (exp(-cluster_size / N_bus)).
    """
    if "res_bus" not in net:
        return np.nan

    res_bus = net.res_bus
    abnormal = res_bus.index[np.abs(res_bus.vm_pu - 1.0) > 0.05].tolist()
    if not abnormal:
        return 1.0

    G = nx.Graph()
    for _, row in net.line.iterrows():
        if row["in_service"]:
            G.add_edge(int(row["from_bus"]), int(row["to_bus"]))

    sub_nodes = [n for n in abnormal if n in G]
    if not sub_nodes:
        return 1.0

    components = list(nx.connected_components(G.subgraph(sub_nodes)))
    largest_cluster = max(len(c) for c in components) if components else 0
    P_V = largest_cluster / len(net.bus)
    return np.exp(-P_V)


def generalized_power_mean(values, weights, p=1):
    """Weighted power mean (p=1: weighted arithmetic mean; p→0 → geometric)."""
    values = np.array(values, dtype=float)
    weights = np.array(weights, dtype=float)
    if np.any(np.isnan(values)):
        return np.nan
    if p == 0:
        return np.prod(values ** weights)
    return (np.sum(weights * (values ** p))) ** (1 / p)


In [ ]:
def create_cyber_network_from_pmus(net, num_pmUs):
    """
    Create a cyber control graph based on PMU placement.
    PMUs are placed at high-degree buses (most connected), and
    the control graph is the subgraph induced by PMU buses.
    """
    in_service_buses = net.bus[net.bus["in_service"]].index.tolist()
    if len(in_service_buses) == 0:
        print("⚠️ Warning: No in-service buses found!")
        return nx.Graph(), []

    # Degree-based PMU placement
    bus_degrees = {}
    for bus in in_service_buses:
        degree = len(net.line[(net.line["from_bus"] == bus) |
                              (net.line["to_bus"] == bus)])
        bus_degrees[bus] = degree

    sorted_buses = sorted(bus_degrees.items(),
                          key=lambda x: x[1],
                          reverse=True)
    pmu_buses = [bus for bus, _ in sorted_buses[:min(num_pmUs, len(in_service_buses))]]

    control_graph = nx.Graph()
    control_graph.add_nodes_from(pmu_buses)

    # Only connect PMU buses along active lines
    for _, row in net.line.iterrows():
        if row["in_service"]:
            from_bus, to_bus = row["from_bus"], row["to_bus"]
            if from_bus in pmu_buses and to_bus in pmu_buses:
                control_graph.add_edge(from_bus, to_bus)

    return control_graph, pmu_buses


def compute_cyber_reachability(control_graph, failed_nodes):
    """C_t = fraction of control nodes reachable from the control center."""
    total_nodes = len(control_graph.nodes)
    if total_nodes == 0:
        return 1.0

    working_graph = control_graph.copy()
    working_graph.remove_nodes_from(failed_nodes)

    if len(working_graph.nodes) == 0:
        return 0.0

    # Define the control center as the PMU with smallest bus index
    control_center = min(control_graph.nodes())

    if control_center in failed_nodes or control_center not in working_graph:
        return 0.0

    try:
        reachable_nodes = nx.node_connected_component(working_graph, control_center)
        return len(reachable_nodes) / total_nodes
    except Exception:
        return 0.0


In [ ]:
def solve_detection_optimization(net, control_graph, failed_nodes, pmu_buses):
    """
    Detection time T_det (hours) based on:
      - cyber reachability C_t
      - PMU coverage (fraction of buses with PMUs)
    Returns np.inf if detection is effectively impossible.
    """
    C_t = compute_cyber_reachability(control_graph, failed_nodes)
    if C_t < 0.3:
        return np.inf

    total_buses = len(net.bus)
    pmu_coverage = len(pmu_buses) / total_buses if total_buses > 0 else 0

    T_min_detect = 0.25  # minimum detection time (15 min)
    observability_score = C_t * pmu_coverage

    if observability_score < 0.1:
        return np.inf

    # Worse observability => longer delay (scaled)
    delay_factor = (1.0 - observability_score) * 2.0
    T_det = T_min_detect + delay_factor
    return T_det

In [ ]:
def solve_response_optimization(net, t_attack, t_detect,
                                gamma_param=0.85,
                                epsilon_param=0.10,
                                cyber_reachability=1.0):
    """
    Response phase surrogate optimization (via CVXPY).
    Maximize served load subject to:
      - minimum load service ratio gamma_param
      - generator Pmin/Pmax
      - total P balance
    Returns an effective response time T_resp in hours (heuristic).
    """
    try:
        n_gens = len(net.gen) if "gen" in net else 0
        n_loads = len(net.load) if "load" in net else 0
        if n_gens == 0 or n_loads == 0:
            return 1.0

        P_gen = cp.Variable(n_gens)
        P_load = cp.Variable(n_loads)

        P_gen_max = net.gen["max_p_mw"].values
        P_gen_min = net.gen["min_p_mw"].values
        P_load_nom = net.load["p_mw"].values

        objective = cp.Maximize(cp.sum(P_load))

        constraints = [
            cp.sum(P_load) >= gamma_param * np.sum(P_load_nom),
            P_gen >= P_gen_min,
            P_gen <= P_gen_max,
            P_load >= 0,
            P_load <= P_load_nom,
            cp.sum(P_gen) == cp.sum(P_load),
        ]

        problem = cp.Problem(objective, constraints)
        problem.solve(solver=cp.ECOS, verbose=False)

        if problem.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
            T_resp_base = 0.5  # base response time
            cyber_delay = (1.0 - cyber_reachability) * 1.0

            load_served = cp.sum(P_load).value or 0.0
            load_shed_fraction = 1.0 - (load_served / np.sum(P_load_nom))
            shedding_delay = load_shed_fraction * 0.5

            return T_resp_base + cyber_delay + shedding_delay
        else:
            return 4.0  # worst-case fallback

    except Exception as e:
        print(f" Response optimization failed: {e}")
        return 2.0


In [ ]:
def solve_recovery_optimization(net, control_graph, failed_nodes, T_resp,
                                Gamma_param=0.95, Beta_param=0.85,
                                epsilon_param=0.05, failed_lines=None):
    """
    Recovery phase surrogate optimization.
    Combines:
      - cyber repair time
      - physical line repair time
      - load restoration optimization
    Returns T_rec in hours (heuristic).
    """
    try:
        # --- Cyber repair time ---
        n_failed_cyber = len(failed_nodes)
        if n_failed_cyber == 0:
            T_cyber_recovery = 0.25
        else:
            repair_time_per_node = 0.5
            max_parallel_cyber_repairs = 3
            total_nodes = len(control_graph.nodes)
            nodes_needed = int(np.ceil(Beta_param * total_nodes))
            nodes_to_repair = min(nodes_needed, n_failed_cyber)
            T_cyber_recovery = (nodes_to_repair * repair_time_per_node) / max_parallel_cyber_repairs

        # --- Physical repair time ---
        n_failed_lines = len(failed_lines) if failed_lines else 1
        repair_time_per_line = 2.0
        max_parallel_line_repairs = 2

        repair_already_done = 0.5 * T_resp
        total_physical_repair_time = (n_failed_lines * repair_time_per_line) / max_parallel_line_repairs
        T_physical_recovery = max(0, total_physical_repair_time - repair_already_done)

        # --- Load restoration optimization ---
        n_loads = len(net.load) if "load" in net else 0
        n_gens  = len(net.gen)  if "gen"  in net else 0

        if n_loads > 0 and n_gens > 0:
            try:
                P_load_restored = cp.Variable(n_loads)
                P_gen_dispatch  = cp.Variable(n_gens)

                P_load_nom = net.load["p_mw"].values
                P_gen_max  = net.gen["max_p_mw"].values
                P_gen_min  = net.gen["min_p_mw"].values

                objective = cp.Maximize(cp.sum(P_load_restored))
                constraints = [
                    cp.sum(P_load_restored) >= Gamma_param * np.sum(P_load_nom),
                    P_gen_dispatch >= P_gen_min,
                    P_gen_dispatch <= P_gen_max,
                    P_load_restored >= 0,
                    P_load_restored <= P_load_nom,
                    cp.sum(P_gen_dispatch) == cp.sum(P_load_restored),
                ]

                problem = cp.Problem(objective, constraints)
                problem.solve(solver=cp.ECOS, verbose=False)

                if problem.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
                    # Map (Gamma - gamma) gap to an extra delay term (simple heuristic)
                    T_load_restoration = (Gamma_param - gamma) / 0.1 * 0.5
                else:
                    T_load_restoration = 1.0
            except Exception:
                T_load_restoration = 0.5
        else:
            T_load_restoration = 0.5

        # Hold time after main restoration
        T_hold = 1.0

        # Parallel and sequential portions of recovery
        parallel_phase   = max(T_cyber_recovery, T_physical_recovery * 0.7)
        sequential_phase = T_physical_recovery * 0.3 + T_load_restoration + T_hold

        T_rec = parallel_phase + sequential_phase
        T_rec = max(T_rec, 1.5)
        return T_rec

    except Exception as e:
        print(f" Recovery optimization failed: {e}")
        return max(len(failed_nodes) * 0.5 + 2.0, 2.0)


In [ ]:
def compute_cyber_score(C_t, T_det, T_resp, T_rec):
    """
    Cyber score:
    S_C = C_t^u1 * exp(-T_det/Tdet_max)^u2 *
          exp(-T_resp/Tresp_max)^u3 * exp(-T_rec/Trec_max)^u4
    """
    if T_det is None or np.isinf(T_det):
        return 0.0

    T_det_use  = min(T_det  if T_det  is not None else 0.0, Tdet_max)
    T_resp_use = min(T_resp if T_resp is not None else 0.0, Tresp_max)
    T_rec_use  = min(T_rec  if T_rec  is not None else 0.0, Trec_max)

    term1 = C_t ** u1
    term2 = np.exp(-T_det_use  / Tdet_max)  ** u2
    term3 = np.exp(-T_resp_use / Tresp_max) ** u3
    term4 = np.exp(-T_rec_use  / Trec_max)  ** u4

    return term1 * term2 * term3 * term4


def compute_R_t(SP, SC, eta=0.2):
    """
    System-level resilience:
      R(t) = SP(t) * [η + (1-η) * √SC(t)]
    """
    if np.isnan(SP) or np.isnan(SC):
        return np.nan
    return SP * (eta + (1 - eta) * np.sqrt(SC))


In [ ]:
# =====================================================================================
# 5. MAIN SIMULATION LOOP OVER SCENARIOS
# =====================================================================================

for s_idx in range(1, NUM_SCENARIOS + 1):
    # Restore original in_service flags
    net.line["in_service"] = original_line_in_service.copy()
    if original_gen_in_service is not None:
        net.gen["in_service"] = original_gen_in_service.copy()
    if original_load_in_service is not None:
        net.load["in_service"] = original_load_in_service.copy()

    # Scenario directories
    scenario_root = os.path.join(base_root, f"scenario{s_idx}")
    os.makedirs(scenario_root, exist_ok=True)
    os.makedirs(os.path.join(scenario_root, "logs"), exist_ok=True)
    os.makedirs(os.path.join(scenario_root, "power_flow_results"), exist_ok=True)

    # Random attack time and line (within 20–40% of horizon)
    trigger_step    = random.randint(int(0.2 * n_ts), int(0.4 * n_ts))
    target_line_idx = random.choice(element_indices)

    meta_rows = [{
        "scenario": s_idx,
        "target_element": target_element,
        "target_index": int(target_line_idx),
        "planned_trigger_step": int(trigger_step),
        "attack_fired": False,
        "T_det_hours": None,
        "T_resp_hours": None,
        "T_rec_hours": None,
    }]

    failed_nodes = []
    failed_lines = []
    resilience_records = []
    attack_fired = False

    # Initialize T_det, T_resp, T_rec before attack
    T_det  = T_det_baseline
    T_resp = 0.0
    T_rec  = 0.0

    for t, ts in enumerate(tqdm(time_index, desc=f"Scenario {s_idx}", leave=False)):
        # --------------------------------------------------
        # Attack trigger – physical line outage + cyber node failures
        # --------------------------------------------------
        if t == trigger_step and not attack_fired:
            if net.line.at[target_line_idx, "in_service"]:
                # Physical line outage
                net.line.at[target_line_idx, "in_service"] = False
                failed_lines = [target_line_idx]

                # Cyber node failures: 20% of PMUs
                num_failed = max(1, int(0.2 * len(control_graph.nodes())))
                failed_nodes = random.sample(list(control_graph.nodes()), k=num_failed)

                attack_fired = True
                meta_rows[0]["attack_fired"] = True

                print(f"\n  ⚡ Attack in Scenario {s_idx} at t={t}: "
                      f"Line {target_line_idx}, {num_failed} cyber nodes")

                # Detection time
                print("  🔍 Computing T_det...")
                T_det = solve_detection_optimization(net, control_graph, failed_nodes, pmu_buses)
                meta_rows[0]["T_det_hours"] = float(T_det) if not np.isinf(T_det) else None

                if not np.isinf(T_det):
                    print(f"     T_det = {T_det:.2f} hours")

                    # Response time
                    print("  ⚙️ Computing T_resp...")
                    C_t_now = compute_cyber_reachability(control_graph, failed_nodes)
                    T_resp = solve_response_optimization(
                        net,
                        trigger_step,
                        trigger_step + int(T_det / step_hours),
                        gamma_param=gamma,
                        epsilon_param=epsilon,
                        cyber_reachability=C_t_now,
                    )
                    meta_rows[0]["T_resp_hours"] = float(T_resp)
                    print(f"     T_resp = {T_resp:.2f} hours")

                    # Recovery time
                    print("  🔄 Computing T_rec...")
                    T_rec = solve_recovery_optimization(
                        net,
                        control_graph,
                        failed_nodes,
                        T_resp,
                        Gamma_param=Gamma,
                        Beta_param=Beta,
                        epsilon_param=epsilon_rec,
                        failed_lines=failed_lines,
                    )
                    meta_rows[0]["T_rec_hours"] = float(T_rec)
                    print(f"     T_rec = {T_rec:.2f} hours")
                    print(f"     TOTAL = {T_det + T_resp + T_rec:.2f} hours\n")
                else:
                    print("     Attack undetectable!\n")
                    T_resp = Tresp_max
                    T_rec  = Trec_max

        # --------------------------------------------------
        # Power flow for this time step
        # --------------------------------------------------
        try:
            pp.runpp(net, algorithm="nr", calculate_voltage_angles=True,
                     enforce_q_lims=True)
            converged = net.converged
        except Exception:
            converged = False

        if not converged:
            SP_t = SC_t = R_t = LSR_t = f2_t = C_t_val = np.nan
        else:
            net = add_bus_information(net)

            # Physical layer: LSR + voltage persistence
            LSR_t = compute_load_served_ratio_from_results(net)
            f2_t  = compute_voltage_persistence_from_results(net)
            SP_t  = generalized_power_mean([LSR_t, f2_t],
                                           weights=[0.6, 0.4], p=1)

            # Cyber layer: reachability
            C_t_val = compute_cyber_reachability(control_graph, failed_nodes)

            # Cyber score and resilience
            SC_t = compute_cyber_score(C_t_val, T_det, T_resp, T_rec)
            R_t  = compute_R_t(SP_t, SC_t)

        resilience_records.append({
            "time": ts,
            "timestep": t,
            "LSR": LSR_t,
            "Voltage_Persistence": f2_t,
            "SP": SP_t,
            "C_t": C_t_val,
            "T_det": T_det,
            "T_resp": T_resp,
            "T_rec": T_rec,
            "SC": SC_t,
            "R": R_t,
            "converged": converged,
            "attack_active": attack_fired,
        })

    # ------------------------------------------------------
    # Save results for this scenario
    # ------------------------------------------------------
    resilience_df = pd.DataFrame(resilience_records)
    resilience_df.to_csv(os.path.join(scenario_root, "resilience_timeseries.csv"),
                         index=False)
    pd.DataFrame(meta_rows).to_csv(
        os.path.join(scenario_root, "logs", "scenario_meta.csv"),
        index=False,
    )

    pf_results_dir = os.path.join(scenario_root, "power_flow_results")
    if "res_load" in net:
        net.res_load.to_csv(os.path.join(pf_results_dir, "res_load.csv"))
    if "res_line" in net:
        net.res_line.to_csv(os.path.join(pf_results_dir, "res_line.csv"))
    if "res_bus" in net:
        net.res_bus.to_csv(os.path.join(pf_results_dir, "res_bus.csv"))
    if "res_gen" in net:
        net.res_gen.to_csv(os.path.join(pf_results_dir, "res_gen.csv"))
    if "res_ext_grid" in net:
        net.res_ext_grid.to_csv(os.path.join(pf_results_dir, "res_ext_grid.csv"))
    if "res_trafo" in net:
        net.res_trafo.to_csv(os.path.join(pf_results_dir, "res_trafo.csv"))

    # PMU configuration (same for all scenarios, stored per scenario)
    pmu_config = pd.DataFrame({
        "pmu_bus": pmu_buses,
        "is_control_center": [bus == min(pmu_buses) for bus in pmu_buses]
                              if len(pmu_buses) > 0 else []
    })
    pmu_config.to_csv(os.path.join(scenario_root, "logs", "pmu_configuration.csv"),
                      index=False)

    print(f"Scenario {s_idx} completed\n")


In [ ]:
# =====================================================================================
# 6. RESTORE STATE AND ARCHIVE RESULTS
# =====================================================================================

net.line["in_service"] = original_line_in_service.copy()
if original_gen_in_service is not None:
    net.gen["in_service"] = original_gen_in_service.copy()
if original_load_in_service is not None:
    net.load["in_service"] = original_load_in_service.copy()

# Zip all results
zip_path = shutil.make_archive(case_name, "zip", root_dir=".", base_dir=case_name)
print(f"Done. All scenarios saved in '{case_name}.zip'")


### Test case 2_N-2 on transmission lines and DOS on PMU nodes 

In [ ]:
import os, shutil, random
import numpy as np
import pandas as pd
import pandapower as pp
import networkx as nx
import cvxpy as cp
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import copy
 
# =====================================================================================
# ---------------------- CONFIGURATION ----------------------
# =====================================================================================
 
case_name = "case_rm2"
n_ts = 96
step_hours = 0.25
rng = np.random.default_rng(0)
 
# Parallelization parameters
MAX_WORKERS = 6        # number of CPU cores to use
BATCH_SIZE = 10        # number of parallel scenarios per batch
 
# Load network
# net = your_network_here  # Uncomment and load your network
 
# Create single base directory
base_root = f"./{case_name}"
if os.path.exists(base_root):
    shutil.rmtree(base_root)  # Remove old case_rm2 folder
os.makedirs(base_root, exist_ok=True)
 
# Parameters
u1, u2, u3, u4 = 0.3, 0.25, 0.25, 0.2
gamma, epsilon, Beta, Gamma, epsilon_rec = 0.85, 0.10, 0.85, 0.95, 0.05
Tdet_max, Tresp_max, Trec_max = 6.0, 4.0, 12.0
 
# Store original state
original_line_in_service = net.line["in_service"].copy()
original_gen_in_service = net.gen["in_service"].copy()
original_load_in_service = net.load["in_service"].copy()
 
element_indices = list(net.line.index)
num_scenarios = int(input("Enter number of N-2 scenarios to run: "))
 
time_index = pd.date_range("2025-01-01", periods=n_ts, freq=pd.Timedelta(hours=step_hours))
 
# =====================================================================================
# ---------------------- HELPER FUNCTIONS ----------------------
# =====================================================================================
 
def add_bus_information(net):
    """Add bus information to result dataframes"""
    mappings = [
        ("res_line", "line", ["from_bus", "to_bus", "in_service"]),
        ("res_gen", "gen", ["bus", "in_service"]),
        ("res_load", "load", ["bus", "in_service"]),
        ("res_ext_grid", "ext_grid", ["bus", "in_service"]),
        ("res_trafo", "trafo", ["hv_bus", "lv_bus", "in_service"]),
    ]
 
    for res_name, base_name, cols in mappings:
        if hasattr(net, res_name) and hasattr(net, base_name):
            res_df = getattr(net, res_name)
            base_df = getattr(net, base_name)
 
            merged = res_df.merge(
                base_df[cols],
                left_index=True,
                right_index=True,
                how="left",
                suffixes=("", "_y")
            )
            setattr(net, res_name, merged.loc[:, ~merged.columns.str.endswith("_y")])
 
    return net
 
 
def compute_load_served_ratio_from_results(net):
    """Compute LSR"""
    if not ("res_load" in net and "load" in net and "line" in net):
        return np.nan
 
    active_lines = net.line[net.line["in_service"]]
    active_buses = set(active_lines["from_bus"]).union(set(active_lines["to_bus"]))
 
    if "bus" not in net.res_load.columns:
        net = add_bus_information(net)
 
    served_loads = net.res_load[net.res_load["bus"].isin(active_buses)]
    served = served_loads["p_mw"].sum()
    demand = net.load["p_mw"].sum()
 
    return np.nan if demand <= 0 else min(served / demand, 1.0)
 
 
def compute_voltage_persistence_from_results(net):
    """Compute voltage persistence"""
    if "res_bus" not in net:
        return np.nan
 
    res_bus = net.res_bus
    abnormal = res_bus.index[np.abs(res_bus.vm_pu - 1.0) > 0.05].tolist()
 
    if not abnormal:
        return 1.0
 
    G = nx.Graph()
    for _, row in net.line.iterrows():
        if row["in_service"]:
            G.add_edge(int(row["from_bus"]), int(row["to_bus"]))
 
    sub_nodes = [n for n in abnormal if n in G]
    if not sub_nodes:
        return 1.0
 
    components = list(nx.connected_components(G.subgraph(sub_nodes)))
    largest_cluster = max(len(c) for c in components) if components else 0
    P_V = largest_cluster / len(net.bus)
 
    return np.exp(-P_V)
 
 
def generalized_power_mean(values, weights, p=1):
    """Weighted power mean"""
    values, weights = np.array(values, dtype=float), np.array(weights, dtype=float)
    if np.any(np.isnan(values)):
        return np.nan
    if p == 0:
        return np.prod(values ** weights)
    return (np.sum(weights * (values ** p))) ** (1 / p)
 
 
def compute_cyber_reachability(control_graph, failed_nodes):
    """Compute cyber reachability"""
    total_nodes = len(control_graph.nodes)
    if total_nodes == 0:
        return 1.0
 
    working_graph = control_graph.copy()
    working_graph.remove_nodes_from(failed_nodes)
 
    if len(working_graph.nodes) == 0:
        return 0.0
 
    control_center = min(control_graph.nodes())
    if control_center in failed_nodes or control_center not in working_graph:
        return 0.0
 
    try:
        reachable_nodes = nx.node_connected_component(working_graph, control_center)
        return len(reachable_nodes) / total_nodes
    except:
        return 0.0
 
 
def compute_cyber_score(C_t, T_det, T_resp, T_rec):
    """Compute cyber score"""
    if T_det is None or np.isinf(T_det):
        return 0.0
 
    T_det_use = min(T_det if T_det is not None else 0.0, Tdet_max)
    T_resp_use = min(T_resp if T_resp is not None else 0.0, Tresp_max)
    T_rec_use = min(T_rec if T_rec is not None else 0.0, Trec_max)
 
    term1 = C_t ** u1
    term2 = np.exp(-T_det_use / Tdet_max) ** u2
    term3 = np.exp(-T_resp_use / Tresp_max) ** u3
    term4 = np.exp(-T_rec_use / Trec_max) ** u4
 
    return term1 * term2 * term3 * term4
 
 
def compute_R_t(SP, SC, eta=0.2):
    """Compute system resilience"""
    if np.isnan(SP) or np.isnan(SC):
        return np.nan
    return SP * (eta + (1 - eta) * np.sqrt(SC))
 
 
def save_power_flow_results(net, output_dir, prefix=""):
    """Save all power flow results to CSV"""
    os.makedirs(output_dir, exist_ok=True)
 
    result_tables = {
        "res_bus": "bus_results",
        "res_line": "line_results",
        "res_load": "load_results",
        "res_gen": "gen_results",
        "res_ext_grid": "ext_grid_results",
        "res_trafo": "trafo_results",
    }
 
    for attr, filename in result_tables.items():
        if hasattr(net, attr):
            df = getattr(net, attr)
            filepath = os.path.join(output_dir, f"{prefix}{filename}.csv")
            df.to_csv(filepath)
 
    # Also save network state
    state_tables = {
        "bus": "bus_state",
        "line": "line_state",
        "load": "load_state",
        "gen": "gen_state",
        "ext_grid": "ext_grid_state",
        "trafo": "trafo_state",
    }
 
    for attr, filename in state_tables.items():
        if hasattr(net, attr):
            df = getattr(net, attr)
            filepath = os.path.join(output_dir, f"{prefix}{filename}.csv")
            df.to_csv(filepath)
 
 
# =====================================================================================
# ---------------------- SINGLE SCENARIO FUNCTION ----------------------
# =====================================================================================
 
def simulate_scenario(s_idx, net_base, time_index):
    """Simulate a single N-2 contingency scenario"""
 
    # Deep copy network
    local_net = copy.deepcopy(net_base)
 
    # Create scenario folder structure
    scenario_root = os.path.join(base_root, f"scenario{s_idx}")
    os.makedirs(scenario_root, exist_ok=True)
 
    logs_dir = os.path.join(scenario_root, "logs")
    pf_before_dir = os.path.join(scenario_root, "power_flow_before_attack")
    pf_after_dir = os.path.join(scenario_root, "power_flow_after_attack")
 
    os.makedirs(logs_dir, exist_ok=True)
    os.makedirs(pf_before_dir, exist_ok=True)
    os.makedirs(pf_after_dir, exist_ok=True)
 
    # Randomly choose 2 different lines for N-2 contingency
    failed_lines = random.sample(element_indices, 2)
    trigger_step = random.randint(int(0.2 * n_ts), int(0.4 * n_ts))
 
    # Metadata
    meta_data = {
        "scenario": s_idx,
        "failed_line_1": int(failed_lines[0]),
        "failed_line_2": int(failed_lines[1]),
        "attack_trigger_step": int(trigger_step),
        "attack_trigger_time": str(time_index[trigger_step]),
        "total_timesteps": n_ts,
        "step_hours": step_hours,
    }
 
    attack_fired = False
    resilience_records = []
 
    # Baseline before attack
    C_t, T_det, T_resp, T_rec = 1.0, 0.25, 0.0, 0.0
 
    # Run initial power flow and save BEFORE state
    try:
        pp.runpp(local_net, algorithm="nr", max_iteration=30)
        save_power_flow_results(local_net, pf_before_dir, prefix="initial_")
        meta_data["initial_pf_converged"] = True
    except Exception as e:
        meta_data["initial_pf_converged"] = False
        meta_data["initial_pf_error"] = str(e)
 
    for t, ts in enumerate(time_index):
 
        # Trigger N-2 attack
        if t == trigger_step and not attack_fired:
            for line_idx in failed_lines:
                if local_net.line.at[line_idx, "in_service"]:
                    local_net.line.at[line_idx, "in_service"] = False
 
            attack_fired = True
            meta_data["attack_fired"] = True
 
            # Save power flow results immediately AFTER attack
            try:
                pp.runpp(local_net, algorithm="nr", max_iteration=30)
                save_power_flow_results(local_net, pf_after_dir, prefix="post_attack_")
                meta_data["post_attack_pf_converged"] = True
            except Exception as e:
                meta_data["post_attack_pf_converged"] = False
                meta_data["post_attack_pf_error"] = str(e)
 
        # Run power flow for current timestep
        try:
            pp.runpp(local_net, algorithm="nr", max_iteration=30)
            converged = local_net.converged
        except Exception:
            converged = False
 
        # Compute metrics
        if converged:
            local_net = add_bus_information(local_net)
 
            LSR_t = compute_load_served_ratio_from_results(local_net)
            f2_t = compute_voltage_persistence_from_results(local_net)
            SP_t = generalized_power_mean([LSR_t, f2_t], [0.6, 0.4])
 
            # Simplified cyber degradation model
            if attack_fired:
                C_t = max(0.1, 1.0 - 0.2)  # 20% degradation after attack
            else:
                C_t = 1.0
 
            SC_t = compute_cyber_score(C_t, T_det, T_resp, T_rec)
            R_t = compute_R_t(SP_t, SC_t)
        else:
            LSR_t = f2_t = SP_t = C_t = SC_t = R_t = np.nan
 
        # Store timestep results
        resilience_records.append({
            "time": ts,
            "timestep": t,
            "LSR": LSR_t,
            "Voltage_Persistence": f2_t,
            "SP": SP_t,
            "C_t": C_t,
            "SC": SC_t,
            "R": R_t,
            "converged": converged,
            "attack_active": attack_fired,
        })
 
    # Save resilience timeseries
    resilience_df = pd.DataFrame(resilience_records)
    resilience_df.to_csv(os.path.join(scenario_root, "resilience_timeseries.csv"), index=False)
 
    # Save metadata/logs
    meta_df = pd.DataFrame([meta_data])
    meta_df.to_csv(os.path.join(logs_dir, "scenario_metadata.csv"), index=False)
 
    # Save summary statistics
    if len(resilience_df) > 0:
        summary = {
            "scenario": s_idx,
            "min_R": resilience_df["R"].min(),
            "max_R": resilience_df["R"].max(),
            "mean_R": resilience_df["R"].mean(),
            "min_LSR": resilience_df["LSR"].min(),
            "convergence_rate": resilience_df["converged"].sum() / len(resilience_df),
        }
        summary_df = pd.DataFrame([summary])
        summary_df.to_csv(os.path.join(logs_dir, "summary_statistics.csv"), index=False)
 
    return s_idx, meta_data
 
 
# =====================================================================================
# ---------------------- PARALLEL BATCH EXECUTION ----------------------
# =====================================================================================
 
def run_in_batches():
    """Run scenarios in batches with progress tracking"""
    all_indices = list(range(1, num_scenarios + 1))
    completed_scenarios = []
 
    print(f"\n{'='*60}")
    print(f"Starting N-2 Contingency Analysis")
    print(f"{'='*60}")
    print(f"Total scenarios: {num_scenarios}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Workers per batch: {MAX_WORKERS}")
    print(f"Results folder: {base_root}")
    print(f"{'='*60}\n")
 
    for batch_num, b in enumerate(range(0, len(all_indices), BATCH_SIZE), 1):
        batch = all_indices[b:b + BATCH_SIZE]
 
        print(f"\n🔄 Processing Batch {batch_num}/{(len(all_indices)-1)//BATCH_SIZE + 1}")
        print(f"   Scenarios: {batch[0]} to {batch[-1]}")
 
        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(simulate_scenario, s, net, time_index): s
                for s in batch
            }
 
            for future in tqdm(as_completed(futures), total=len(batch),
                             desc=f"Batch {batch_num}"):
                try:
                    s_idx, meta = future.result()
                    completed_scenarios.append(meta)
                    # print(f"   ✅ Scenario {s_idx} completed")
                except Exception as e:
                    s_idx = futures[future]
                    print(f"   ❌ Scenario {s_idx} failed: {e}")
 
    # Save overall summary
    if completed_scenarios:
        overall_summary = pd.DataFrame(completed_scenarios)
        overall_summary.to_csv(os.path.join(base_root, "all_scenarios_summary.csv"), index=False)
        print(f"\n✅ Overall summary saved to '{base_root}/all_scenarios_summary.csv'")
 
    return completed_scenarios
 
 
# =====================================================================================
# ---------------------- RUN SIMULATION ----------------------
# =====================================================================================
 
completed = run_in_batches()
 
# =====================================================================================
# ---------------------- ZIP RESULTS ----------------------
# =====================================================================================
 
print(f"\n{'='*60}")
print("Creating archive...")
print(f"{'='*60}")
 
zip_path = shutil.make_archive(case_name, "zip", root_dir=".", base_dir=case_name)
 
print(f"\n✅ ALL {num_scenarios} N-2 SCENARIOS COMPLETED!")
print(f"{'='*60}")
print(f"Results archived to: '{zip_path}'")
print(f"Successful scenarios: {len(completed)}/{num_scenarios}")
print(f"{'='*60}\n")
 
files.download(zip_path)